# Probability in machine learning

## Why this matters

Section 03 uses squared error for regression and cross-entropy for classification, and it is easy to treat them as formulas handed down from nowhere. They are not arbitrary. Each loss is the negative log of a probability model for how the data was generated. Choose the noise model and the loss follows.

Think of fitting a line through scattered points. Assuming the scatter is gentle and bell-shaped gives squared error, the familiar least-squares fit. Assume the scatter has heavier tails and you get absolute error, which ignores outliers instead of chasing them.

Two knobs. Maximum likelihood ties a loss to a noise model: pick the parameters that make the observed data most probable, take the negative log, and you are minimising a loss. And the regularisation strength $\lambda=\sigma^2/\tau^2$ comes from a prior: a tighter prior (smaller $\tau$) means a larger $\lambda$ and weights pulled harder towards zero; let $\tau\to\infty$ and $\lambda\to0$, no penalty.

Order: intuition, maths, idea, from scratch, library, what breaks.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)

from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import log_loss

## 1. Intuition

Start by writing down a probability model for how the data was produced, noise and all. Maximum likelihood estimation (MLE) then picks the parameters that make the data you actually observed as probable as possible. Turn that goal upside down, take the negative log, and maximising likelihood becomes minimising a loss. The log turns the product over data points into a sum, which is why every loss you have seen is a sum over examples.

So a loss function is not a free choice dressed up as maths. Choosing a loss is choosing an assumption about the noise. Squared error assumes Gaussian noise. Absolute error assumes heavier-tailed Laplace noise. Picking the wrong one is picking the wrong story about your data, and the fit suffers for it.

## 2. Maths

For independent data, the likelihood is a product, and the log turns it into a sum:

$$\hat\theta=\arg\max_\theta\prod_i p(y_i\mid x_i,\theta)=\arg\min_\theta\left[-\sum_i\log p(y_i\mid x_i,\theta)\right]$$

**Gaussian noise gives squared error.** If $y=f(x)+\varepsilon$ with $\varepsilon\sim N(0,\sigma^2)$:

$$-\log p(y\mid x)=\frac{(y-f(x))^2}{2\sigma^2}+\log(\sigma\sqrt{2\pi})$$

Summed over the data and with $\sigma$ fixed, this is the sum of squared errors times a constant.

**Bernoulli gives cross-entropy.** If $y\in\{0,1\}$ and $P(y=1)=\hat p$:

$$-\log p(y)=-\left[y\log\hat p+(1-y)\log(1-\hat p)\right]$$

**Categorical gives softmax cross-entropy.** With $K$ classes, $\hat p=\mathrm{softmax}(z)$, $\hat p_k=e^{z_k}/\sum_j e^{z_j}$ and the loss is $-\log\hat p_{y}$.

**A prior gives regularisation.** MAP estimation maximises likelihood times prior, so it adds $-\log p(\theta)$ to the loss. A Gaussian prior $\theta\sim N(0,\tau^2)$ adds $\|\theta\|^2/(2\tau^2)$, which is an L2 penalty.

## 3. Gaussian noise gives squared error

Generate data from a line with Gaussian noise. Minimise the negative log-likelihood numerically and compare with the least-squares solution from section 01.

In [ ]:
# Sample data: 200 points from the line y = 2x + 1 with Gaussian noise (sd 1.5).
# x is uniform on [-3, 3]; y is a 1-D float array. The true slope 2 and intercept
# 1 are known, so the MLE fit below can be checked against them. Noise sd 1.5 is
# large enough to be visible but small enough to recover the line.
# Seed is fixed so the fitted parameters reproduce.
n = 200
x = rng.uniform(-3, 3, n)
y = 2.0 * x + 1.0 + rng.normal(0, 1.5, n)
A = np.column_stack([x, np.ones(n)])


def gaussian_nll(params: np.ndarray) -> float:
    w, b, log_sigma = params
    sigma = np.exp(log_sigma)
    resid = y - (w * x + b)
    return float(np.sum(resid**2 / (2 * sigma**2) + np.log(sigma * np.sqrt(2 * np.pi))))


fit = minimize(gaussian_nll, x0=[0.0, 0.0, 0.0])
ls = np.linalg.lstsq(A, y, rcond=None)[0]
print("MLE (w, b)    :", fit.x[:2])
print("least squares :", ls)
print("MLE sigma     :", np.exp(fit.x[2]), " residual sd:", (y - A @ ls).std())

The MLE slope and intercept match least squares to the digit, and the MLE noise level comes out as the residual standard deviation. That is the whole claim made concrete: least squares is maximum likelihood under a Gaussian. Change the noise story and the best fit changes with it. Laplace noise, which has heavier tails, gives absolute error, and the parameter that minimises absolute error is the median rather than the mean. The cell below shows why that matters.

In [ ]:
# Sample data: 100 values near 10 (Gaussian, sd 1) with one wild outlier at 1000.
# A 1-D float array. The outlier is set by hand to show the mean chase it while
# the median barely moves. Seed is fixed so the printed mean reproduces.
sample = np.concatenate([rng.normal(10, 1, 99), [1000.0]])  # one wild outlier
print("mean (Gaussian MLE)  :", sample.mean())
print("median (Laplace MLE) :", np.median(sample))

## 4. Bernoulli gives cross-entropy

Fit a logistic model by minimising the Bernoulli negative log-likelihood. Check the value against sklearn's `log_loss` and the weights against `LogisticRegression` with no penalty.

In [ ]:
# Sample data: 500 rows, 2 float features X ~ N(0, 1). The label y_bin is drawn
# from the true logistic model with weights [1.5, -2.0] and intercept 0.5, so the
# from-scratch fit below should recover those numbers up to sampling noise.
# Seed is fixed so the fitted weights and loss reproduce.
n = 500
X = rng.normal(0, 1, (n, 2))
true_w = np.array([1.5, -2.0])
p_true = 1 / (1 + np.exp(-(X @ true_w + 0.5)))
y_bin = (rng.random(n) < p_true).astype(int)


def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))


def bce(params: np.ndarray) -> float:
    # Clip probabilities to [1e-12, 1-1e-12] before the log. A predicted p of
    # exactly 0 or 1 would give log(0) = -inf and break the optimiser; the clip
    # caps the penalty at about 27.6 per point instead.
    p = np.clip(sigmoid(X @ params[:2] + params[2]), 1e-12, 1 - 1e-12)
    return float(-np.mean(y_bin * np.log(p) + (1 - y_bin) * np.log(1 - p)))


fit = minimize(bce, x0=np.zeros(3))
clf = LogisticRegression(penalty=None).fit(X, y_bin)
print("from scratch:", fit.x)
print("sklearn     :", np.append(clf.coef_[0], clf.intercept_))
print("loss (mine, sklearn log_loss):", bce(fit.x), log_loss(y_bin, clf.predict_proba(X)))

## 5. Softmax is a categorical distribution

Softmax turns $K$ raw scores into $K$ probabilities that sum to 1. The loss is the negative log-probability of the true class. Subtract the max score first so the exponentials cannot overflow.

In [ ]:
def softmax(z: np.ndarray) -> np.ndarray:
    # Subtract the row max before exp. Softmax is unchanged by a constant shift
    # in the scores, so this is safe, and it keeps the largest exponent at e^0=1
    # instead of overflowing. Without it the third row below (~e^1001) returns inf.
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


# Sample data: 3 hand-set score rows, 3 classes each, with labels [0, 1, 1].
# The third row (1000, 1001, 999) is deliberately huge to show the max-subtract
# stops overflow. No randomness, so nothing to seed.
z = np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3], [1000.0, 1001.0, 999.0]])
labels = np.array([0, 1, 1])
p = softmax(z)
print(p)
print("rows sum to:", p.sum(axis=1))
ce = -np.log(p[np.arange(3), labels]).mean()
print("cross-entropy:", ce, " sklearn:", log_loss(labels, p, labels=[0, 1, 2]))

## 6. MAP: a Gaussian prior is L2

The same Bayesian move from notebook 02 turns regularisation into a prior. Put a prior $w\sim N(0,\tau^2)$ on the weights, saying you expect them to be small before seeing data, alongside Gaussian noise of variance $\sigma^2$. Maximum a posteriori (MAP) estimation maximises likelihood times prior, which adds $-\log p(w)$ to the loss. For a Gaussian prior that extra term is $\|w\|^2/(2\tau^2)$, an L2 penalty, and the result is exactly ridge regression with $\lambda=\sigma^2/\tau^2$:

$$\hat w=(X^\top X+\lambda I)^{-1}X^\top y$$

Check that against sklearn's `Ridge`, on a small noisy problem with 20 features and only 30 rows where plain least squares overfits badly.

In [ ]:
# Sample data: 30 rows, 20 float features X ~ N(0, 1). Only the first 3 weights
# are non-zero ([3, -2, 1]); the rest are 0. With 20 features and only 30 rows,
# plain least squares overfits, so the MAP/ridge estimate should beat it.
# Noise sd 1.0. Seed is fixed so the weight errors reproduce.
n, d = 30, 20
X = rng.normal(0, 1, (n, d))
w_true = np.zeros(d)
w_true[:3] = [3.0, -2.0, 1.0]
sigma = 1.0
y_reg = X @ w_true + rng.normal(0, sigma, n)

tau = 1.0
lam = sigma**2 / tau**2
w_map = np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y_reg)
w_ridge = Ridge(alpha=lam, fit_intercept=False).fit(X, y_reg).coef_
w_mle = np.linalg.lstsq(X, y_reg, rcond=None)[0]

print("MAP matches Ridge:", np.allclose(w_map, w_ridge))
print("weight error, MLE:", np.linalg.norm(w_mle - w_true))
print("weight error, MAP:", np.linalg.norm(w_map - w_true))

A tighter prior means a smaller $\tau$, a larger $\lambda$, and weights pulled harder towards zero. Let $\tau\to\infty$ and the prior goes flat, $\lambda\to 0$, and MAP collapses back to MLE with no penalty at all. It is the same relationship as the Beta-Binomial in notebook 02: the prior acts as a set of pseudo-observations that holds the estimate down when data is scarce and fades as real data accumulates. Regularisation strength and prior strength are two names for one dial.

In [ ]:
# Sweep the prior width tau over 0.1 to 100 on the same data. The grid spans three
# orders of magnitude so you can see both ends: a tight prior (tau=0.1, large
# lambda) shrinks the weights hard, and a loose prior (tau=100, lambda near 0)
# recovers the overfitting least-squares fit. The error dips in between.
for tau in (0.1, 0.5, 1.0, 5.0, 100.0):
    w = np.linalg.solve(X.T @ X + (sigma**2 / tau**2) * np.eye(d), X.T @ y_reg)
    print(f"tau={tau:>6}: |w|={np.linalg.norm(w):.2f}  error={np.linalg.norm(w - w_true):.2f}")

## 7. What breaks

In [ ]:
# Sample data for two failure cases.
# Separable set: four 1-D points xs = [-2, -1, 1, 2] with labels [0, 0, 1, 1], a
# clean split at 0. Hand-set, no noise, so the logistic MLE can push the weight to
# infinity. Chosen to show why a prior is needed.
# Outlier regression: 50 points on y = 2x (x evenly spaced 0..10) with sd-0.5
# noise, then the last y bumped by +100. Shows one outlier dragging the slope.
# Seed is fixed so the noisy fit reproduces.

# 1. Perfectly separable data: the MLE pushes the weight towards infinity
xs = np.array([-2.0, -1.0, 1.0, 2.0])
ys = np.array([0, 0, 1, 1])


def nll_1d(w: float, prior_tau: float | None = None) -> float:
    p = np.clip(sigmoid(w * xs), 1e-12, 1 - 1e-12)
    loss = -np.sum(ys * np.log(p) + (1 - ys) * np.log(1 - p))
    if prior_tau is not None:
        loss += w**2 / (2 * prior_tau**2)
    return float(loss)


for w in (1, 5, 10, 20):
    print(f"w={w:>2}: NLL={nll_1d(w):.6f}  with prior: {nll_1d(w, 2.0):.3f}")

# 2. Wrong noise model: squared error on data with outliers
x_o = np.linspace(0, 10, 50)
y_o = 2 * x_o + rng.normal(0, 0.5, 50)
y_o[-1] += 100
print("slope, clean fit    :", np.polyfit(x_o[:-1], y_o[:-1], 1)[0])
print("slope, with outlier :", np.polyfit(x_o, y_o, 1)[0])

Without a prior the loss keeps falling as $w$ grows, so on separable data the optimiser chases the weight to infinity and never stops. Add the prior and the loss gains a minimum at a finite weight, which is why a penalty is not optional here. The second failure is the wrong noise model. One outlier in 50 points drags the least-squares slope well off 2, because a Gaussian treats a 100-unit error as essentially impossible and bends the whole fit to explain the thing it believes cannot happen. Match the loss to the noise and the outlier loses its grip.

Section 03 starts from here: pick the noise model, read off the loss, fit the parameters.